# 08 - Comprehensive Scoreboard (Final)

**Inertia v2 - Factor Regimes**

Single head-to-head comparison of every strategy on the same OOS window with
the same monthly-rebalance discipline. Static FF5 is the academic benchmark.
The MPSIF backtest from WRDS (CRSP + Compustat) is included for context as a
real long-only stock-picker.

Window: 2000-02 through 2024-12 (295 months), the intersection of every
strategy's coverage.

Strategies:
1. **Static FF5** (benchmark) - equal-weight FF5 sleeves, rebalanced monthly
2. **Method A** - Markov regime-switching, monthly
3. **Method B** - Ridge regression, monthly
4. **Method C** - Gradient-boosted classifier, monthly
5. **Pure ensemble** - equal-weight average of A, B, C probabilities
6. **Inertia v2** - 50% static FF5 + 50% Method C timed
7. **MPSIF backtest** - 50-stock real strategy from WRDS, monthly rebalance
8. **MPSIF + overlay** - 50% MPSIF + 50% Method C timed FF5 sleeve

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from lib.data import build_factor_panel, FF5_FACTORS
from lib.evaluation import perf_stats, perf_table, sharpe_bootstrap_ci, sharpe_diff_ci, alpha_regression
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'
DATA_DIR = '..'

## 1. Load all data and compute per-strategy returns

In [2]:
# Factor panel (used to compute all FF5-based strategies)
# include_macro=True so the panel covers the same date range as Method B
panel = build_factor_panel(include_macro=True)

# Pre-saved probability matrices from notebooks 02, 03, 04
P_A = pd.read_csv(f'{TABLE_DIR}/05_method_a_probs.csv', index_col=0, parse_dates=True)
P_B = pd.read_csv(f'{TABLE_DIR}/09_method_b_probs.csv', index_col=0, parse_dates=True)
P_C = pd.read_csv(f'{TABLE_DIR}/13_method_c_probs.csv', index_col=0, parse_dates=True)
P_A.index.name = P_B.index.name = P_C.index.name = 'date'

# Snap to calendar month-end so all sources align
for P in (P_A, P_B, P_C):
    P.index = P.index + pd.offsets.MonthEnd(0)

# Real MPSIF strategy from WRDS
mpsif_raw = pd.read_csv(f'{DATA_DIR}/data_input_mpsif_real_returns.csv',
                         parse_dates=['next_date'])
mpsif_raw = mpsif_raw.rename(columns={'next_date': 'date'}).set_index('date')
# CRSP month-ends are last trading day of month; snap to calendar month-end
mpsif_raw.index = mpsif_raw.index + pd.offsets.MonthEnd(0)
r_mpsif_full = mpsif_raw['ret'].sort_index()

print(f'P_A: {P_A.shape}, {P_A.index.min().date()} to {P_A.index.max().date()}')
print(f'P_B: {P_B.shape}, {P_B.index.min().date()} to {P_B.index.max().date()}')
print(f'P_C: {P_C.shape}, {P_C.index.min().date()} to {P_C.index.max().date()}')
print(f'MPSIF: {len(r_mpsif_full)} months, '
      f'{r_mpsif_full.index.min().date()} to {r_mpsif_full.index.max().date()}')


P_A: (434, 5), 1990-01-31 to 2026-02-28
P_B: (312, 5), 2000-02-29 to 2026-01-31
P_C: (312, 5), 2000-02-29 to 2026-01-31
MPSIF: 295 months, 2000-02-29 to 2024-12-31


In [3]:
# Common OOS window: intersection of MPSIF and methods
common = P_A.index.intersection(P_B.index).intersection(P_C.index)
common = common.intersection(r_mpsif_full.index)
common = common.sort_values()
print(f'Common window: {len(common)} months, '
      f'{common.min().date()} to {common.max().date()}')

Common window: 295 months, 2000-02-29 to 2024-12-31


In [4]:
# Helper: convert a probability matrix to monthly composite return
def composite_return(P, mode='linear', cost_bps=5.0):
    """Monthly composite return: equal-weight across 5 timed factor sleeves."""
    P_a = P.reindex(common).dropna(how='any')
    factor_next = panel[FF5_FACTORS].shift(-1).reindex(P_a.index).dropna(how='any')
    P_a = P_a.loc[factor_next.index]
    if mode == 'linear':
        weights = (2 * P_a - 1).clip(-1, 1)
    elif mode == 'soft':
        weights = ((P_a - 0.5) * 4).clip(-1, 1)
    elif mode == 'longflat':
        weights = (P_a > 0.5).astype(float)
    elif mode == 'longonly':
        weights = (2 * P_a - 1).clip(0, 1)
    else:
        raise ValueError(mode)
    sleeves = []
    for f in FF5_FACTORS:
        w = weights[f]
        gross = w * factor_next[f]
        turn = w.diff().abs().fillna(w.abs())
        cost = cost_bps * turn / 1e4
        sleeves.append(gross - cost)
    return pd.DataFrame(sleeves).mean()  # mean across the 5 sleeves

# Build all strategy return series on the common window
factor_next_common = panel[FF5_FACTORS].shift(-1).reindex(common).dropna(how='any')
common = factor_next_common.index  # final common after factor_next dropna
print(f'Final common: {len(common)} months')

r_static = panel[FF5_FACTORS].shift(-1).reindex(common).mean(axis=1)
r_A      = composite_return(P_A, mode='longflat')   # Method A V_lf: longflat (long or cash, no shorts)
r_B      = composite_return(P_B, mode='longflat') # Method B V3: longflat
r_C      = composite_return(P_C, mode='longonly') # Method C V9: longonly
P_ens    = (P_A.reindex(common) + P_B.reindex(common) + P_C.reindex(common)) / 3.0
r_ens    = composite_return(P_ens, mode='longonly')
r_inertia = 0.5 * r_static + 0.5 * r_C  # 50% Static + 50% Method C V9

# Strategy return registry
returns = {
    'Static FF5':      r_static,
    'Method A':        r_A,
    'Method B':        r_B,
    'Method C':        r_C,
    'Pure ensemble':   r_ens,
    'Inertia':         r_inertia,
    'MPSIF backtest':  r_mpsif_full.reindex(common),
    'MPSIF + overlay': (r_mpsif_full.reindex(common) * r_C.reindex(common).apply(lambda x: 1 + x).fillna(1)).fillna(0),
}
# Drop strategies with all NaN
returns = {k: v.dropna() for k, v in returns.items() if v.notna().any()}
for k, v in returns.items():
    print(f'  {k:18s}: {len(v)} months')


Final common: 295 months
  Static FF5        : 295 months
  Method A          : 295 months
  Method B          : 295 months
  Method C          : 295 months
  Pure ensemble     : 295 months
  Inertia           : 295 months
  MPSIF backtest    : 295 months
  MPSIF + overlay   : 295 months


## 2. Headline scoreboard

In [5]:
scoreboard = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(scoreboard, '33_comprehensive_scoreboard', out_dir=TABLE_DIR)
scoreboard

  saved: ../tables/33_comprehensive_scoreboard.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static FF5,295.0000,0.0401,0.0548,0.7310,-0.2105,2.1640,-0.1508
Method A,295.0000,0.0238,0.0380,0.6262,0.5470,2.0700,-0.1011
Method B,295.0000,0.0315,0.0410,0.7680,0.2366,1.4165,-0.0910
Method C,295.0000,0.0104,0.0152,0.6826,1.1803,3.4947,-0.0361
Pure ensemble,295.0000,0.0079,0.0160,0.4969,1.2662,8.4336,-0.0340
Inertia,295.0000,0.0252,0.0325,0.7764,0.0516,1.5413,-0.0910
MPSIF backtest,295.0000,0.1039,0.1937,0.5364,-0.4581,1.9756,-0.5880
MPSIF + overlay,295.0000,0.1037,0.1940,0.5347,-0.4644,1.9848,-0.5873


In [6]:
# Sharpe with bootstrap CI for each
boot = {k: sharpe_bootstrap_ci(v, n_boot=2000) for k, v in returns.items()}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '34_comprehensive_sharpe_ci', out_dir=TABLE_DIR)
boot_df

  saved: ../tables/34_comprehensive_sharpe_ci.{csv,md}


,sharpe,ci_low,ci_high
Static FF5,0.7310,0.2688,1.1918
Method A,0.6262,0.0830,1.0841
Method B,0.7680,0.4057,1.1989
Method C,0.6826,0.2674,1.1718
Pure ensemble,0.4969,-0.0043,0.9745
Inertia,0.7764,0.3176,1.2471
MPSIF backtest,0.5364,0.0461,0.9937
MPSIF + overlay,0.5347,0.0469,0.9926


## 3. Paired Sharpe-difference test vs Static FF5

In [7]:
diffs = {}
for name, r in returns.items():
    if name == 'Static FF5':
        continue
    d = sharpe_diff_ci(r, returns['Static FF5'], n_boot=5000)
    diffs[name] = d
diff_df = pd.DataFrame(diffs).T[['diff','ci_low','ci_high','p_value']]
diff_df['beats_FF5_5pct'] = diff_df['ci_low'] > 0
save_table(diff_df, '35_comprehensive_paired_diff', out_dir=TABLE_DIR)
diff_df

  saved: ../tables/35_comprehensive_paired_diff.{csv,md}


,diff,ci_low,ci_high,p_value,beats_FF5_5pct
Method A,-0.1048,-0.4332,0.1745,0.4492,False
Method B,0.0370,-0.2238,0.4368,0.5344,False
Method C,-0.0484,-0.3978,0.4213,0.9336,False
Pure ensemble,-0.2341,-0.5791,0.1838,0.3560,False
Inertia,0.0455,-0.0247,0.1477,0.1884,False
MPSIF backtest,-0.1946,-0.6783,0.2840,0.4512,False
MPSIF + overlay,-0.1963,-0.6794,0.2815,0.4448,False


## 4. Alpha regression vs FF5 (Newey-West HAC)

In [8]:
factor_panel_for_alpha = panel[FF5_FACTORS + ['RF']].copy()
alpha_rows = {}
for name, r in returns.items():
    try:
        alpha_rows[name] = alpha_regression(r, factor_panel_for_alpha,
                                            FF5_FACTORS, rf_col='RF', hac_lags=6)
    except Exception as e:
        print(f'  {name}: alpha regression failed ({e})')
alpha_df = pd.DataFrame(alpha_rows).T
save_table(alpha_df, '36_comprehensive_alpha_ff5', out_dir=TABLE_DIR)
alpha_df[['alpha_annual','alpha_t','alpha_p','r2'] + [f'{f}_beta' for f in FF5_FACTORS]]

  saved: ../tables/36_comprehensive_alpha_ff5.{csv,md}


,alpha_annual,alpha_t,alpha_p,r2,MKT_RF_beta,SMB_beta,HML_beta,RMW_beta,CMA_beta
Static FF5,0.0192,1.4930,0.1354,0.0186,0.0364,-0.0187,0.0040,-0.0201,0.0789
Method A,-0.0001,-0.0149,0.9881,0.0430,0.0334,0.0059,0.0036,0.0278,0.0809
Method B,0.0088,0.9899,0.3222,0.0206,0.0229,-0.0101,-0.0051,0.0459,0.0438
Method C,-0.0081,-1.6855,0.0919,0.0270,0.0010,-0.0010,-0.0248,0.0031,0.0437
Pure ensemble,-0.0132,-2.9451,0.0032,0.0535,0.0132,0.0006,-0.0023,0.0311,0.0302
Inertia,0.0056,0.7020,0.4827,0.0170,0.0187,-0.0098,-0.0104,-0.0085,0.0613
MPSIF backtest,0.0063,0.2724,0.7853,0.7548,1.0114,0.2076,0.0410,-0.0743,0.3135
MPSIF + overlay,0.0061,0.2623,0.7931,0.7547,1.0129,0.2077,0.0423,-0.0745,0.3116


## 5. Headline cumulative return chart (all 8 strategies)

In [9]:
fig, ax = plt.subplots(figsize=(FULL_COL, 4.4))
color_map = {
    'Static FF5':     C['blue'],
    'Method A':       C['muted'],
    'Method B':       C['green'],
    'Method C':       C['red'],
    'Pure ensemble':  '#888888',
    'Inertia':     C['purple'],
    'MPSIF backtest': '#D08740',
    'MPSIF + overlay':'#000000',
}
lw_map = {
    'Static FF5':1.0, 'Method A':0.6, 'Method B':0.6, 'Method C':0.6,
    'Pure ensemble':0.6, 'Inertia':1.6, 'MPSIF backtest':1.0, 'MPSIF + overlay':1.5,
}
for name in returns:
    cum = (1 + returns[name]).cumprod()
    ax.plot(cum.index, cum.values, color=color_map[name],
            linewidth=lw_map[name], label=name)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Comprehensive scoreboard: all monthly-rebalanced strategies, 2000 to 2024',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=4)
legend_below(ax, ncol=4)
save_fig(fig, '26_comprehensive_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/26_comprehensive_cumret.png


## 6. Sharpe + drawdown bar chart

In [10]:
names = list(returns.keys())
fig, axes = plt.subplots(1, 2, figsize=(FULL_COL, 3.8))
x = np.arange(len(names))

ax = axes[0]
vals = [scoreboard.loc[n, 'sharpe_ann'] for n in names]
bars = ax.bar(x, vals, color=[color_map[n] for n in names], edgecolor='white', linewidth=0.5, width=0.7)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n', 1) for n in names], fontsize=7.5)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Sharpe', loc='left', color=C['ink'])
ax.set_ylim(min(vals) - 0.10, max(vals) + 0.18)
bar_value_labels(ax, bars, fmt='{:+.2f}', offset=0.025, fontsize=8.5, color=C['ink'])

ax = axes[1]
vals = [scoreboard.loc[n, 'max_drawdown'] for n in names]
bars = ax.bar(x, vals, color=[color_map[n] for n in names], edgecolor='white', linewidth=0.5, width=0.7)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n', 1) for n in names], fontsize=7.5)
ax.set_ylabel('Maximum drawdown')
ax.set_title('Max drawdown', loc='left', color=C['ink'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(min(vals) - 0.04, 0.02)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v - 0.01, f'{v:.0%}',
            ha='center', va='top', fontsize=8.5, color=C['red'])
save_fig(fig, '27_comprehensive_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/27_comprehensive_bars.png


## 7. Subsample Sharpe by decade

In [11]:
decades = {
    '2000s':     ('2000-01', '2009-12'),
    '2010s':     ('2010-01', '2019-12'),
    '2020-2024': ('2020-01', '2024-12'),
}
rows = {}
for name, r in returns.items():
    sub = {}
    for label, (lo, hi) in decades.items():
        rr = r.loc[lo:hi].dropna()
        if len(rr) < 12:
            sub[label] = np.nan
        else:
            mu = rr.mean() * 12; sd = rr.std(ddof=1) * np.sqrt(12)
            sub[label] = mu / sd if sd > 0 else np.nan
    rows[name] = sub
decade_sharpe = pd.DataFrame(rows).T
save_table(decade_sharpe, '37_comprehensive_subsample_sharpe', out_dir=TABLE_DIR)
decade_sharpe

  saved: ../tables/37_comprehensive_subsample_sharpe.{csv,md}


,2000s,2010s,2020-2024
Static FF5,1.0824,0.5057,0.5642
Method A,0.9969,0.3451,0.4505
Method B,0.8671,0.5047,0.9947
Method C,1.0665,0.3107,0.6342
Pure ensemble,0.7817,0.1572,0.3623
Inertia,1.1735,0.5009,0.6203
MPSIF backtest,0.2219,0.7750,0.7390
MPSIF + overlay,0.2224,0.7703,0.7365


In [12]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.8))
x = np.arange(len(decades))
n_strats = len(names)
w = 0.78 / n_strats
for i, name in enumerate(names):
    pos = x + (i - (n_strats - 1)/2) * w
    ax.bar(pos, decade_sharpe.loc[name].values, w,
           color=color_map[name], label=name, edgecolor='white', linewidth=0.3)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(list(decades.keys()), fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Sharpe by decade', loc='left', color=C['ink'])
legend_below(ax, ncol=4)
save_fig(fig, '28_comprehensive_subsample_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/28_comprehensive_subsample_bars.png


## 8. Save the full returns matrix for downstream use

In [13]:
all_returns = pd.DataFrame(returns)
all_returns.to_csv(f'{TABLE_DIR}/38_comprehensive_returns.csv')
print(f'Saved: {TABLE_DIR}/38_comprehensive_returns.csv  shape={all_returns.shape}')

Saved: ../tables/38_comprehensive_returns.csv  shape=(295, 8)


## Final findings

All eight strategies on the same 295-month window (2000-02 to 2024-12), all monthly
rebalanced, all benchmarked against static equal-weight FF5.

**Saved artifacts** (slide-builder pulls these directly):

Figures: `26_comprehensive_cumret`, `27_comprehensive_bars`, `28_comprehensive_subsample_bars`

Tables: `33_comprehensive_scoreboard`, `34_comprehensive_sharpe_ci`,
`35_comprehensive_paired_diff`, `36_comprehensive_alpha_ff5`,
`37_comprehensive_subsample_sharpe`, `38_comprehensive_returns.csv`